In [11]:
!pip install opendartreader pandas beautifulsoup4 lxml tqdm


[notice] A new release of pip is available: 25.1.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [12]:
import opendartreader
import pandas as pd

API_KEY = "681ef721403364c216cfc426a7a9147cc826648c"

dart = opendartreader.OpenDartReader(API_KEY)

### 01 Disclosure List Collection

In [ ]:
import requests
import pandas as pd
from tqdm.auto import tqdm
import time

DART_API_KEY = os.getenv("DART_API_KEY")

months = pd.period_range(
    start="2019-01",
    end="2026-4",
    freq="M"
)

all_data = []

for month in tqdm(months):

    bgn_de = month.start_time.strftime("%Y%m%d")
    end_de = month.end_time.strftime("%Y%m%d")
    page_no = 1

    while True:
        url = "https://opendart.fss.or.kr/api/list.json"

        params = {
            "crtfc_key": API_KEY,
            "bgn_de": bgn_de,
            "end_de": end_de,
            "pblntf_ty": "I",
            "page_no": page_no,
            "page_count": 100,
        }

        res = requests.get(url, params=params, timeout=30)
        data = res.json()

        if data.get("status") != "000":
            print(bgn_de, end_de, data.get("status"), data.get("message"))
            break

        rows = data.get("list", [])
        all_data.extend(rows)

        total_page = int(data.get("total_page", 1))

        if page_no >= total_page:
            break

        page_no += 1
        time.sleep(0.1)

dart_list = pd.DataFrame(all_data)

print(dart_list.shape)
dart_list.head()

  0%|          | 0/88 [00:00<?, ?it/s]

(380846, 9)


,corp_code,corp_name,stock_code,corp_cls,report_nm,rcept_no,flr_nm,rcept_dt,rm
0,01168143,케이만금세기차륜집단유한공사,900280,E,[첨부추가]단일판매ㆍ공급계약체결(자율공시)(자회사의 주요경영사항),20190110900105,케이만금세기차륜집단유한공사,20190110,코
1,01168143,케이만금세기차륜집단유한공사,900280,E,[첨부추가]단일판매ㆍ공급계약체결(자율공시)(자회사의 주요경영사항),20190110900100,케이만금세기차륜집단유한공사,20190110,코
2,01168143,케이만금세기차륜집단유한공사,900280,E,[첨부추가]단일판매ㆍ공급계약체결(자율공시)(자회사의 주요경영사항),20190109900120,케이만금세기차륜집단유한공사,20190109,코
3,01168143,케이만금세기차륜집단유한공사,900280,E,[첨부추가]단일판매ㆍ공급계약체결(자율공시)(자회사의 주요경영사항),20190110900102,케이만금세기차륜집단유한공사,20190110,코
4,01168143,케이만금세기차륜집단유한공사,900280,E,[첨부추가]단일판매ㆍ공급계약체결(자회사의 주요경영사항),20190118900054,케이만금세기차륜집단유한공사,20190118,코


In [ ]:
# From dart_list collected with pblntf_ty="I",
# print all unique report_nm values containing earnings-related keywords

mask = dart_list["report_nm"].str.contains("실적|영업|매출|손익", na=False)

print(dart_list[mask]["report_nm"].value_counts().to_string())

report_nm
매출액또는손익구조30%(대규모법인은15%)이상변동                       905
매출액또는손익구조30%(대규모법인은15%)이상변경                       573
연결재무제표기준영업(잠정)실적(공정공시)                            260
영업(잠정)실적(공정공시)                                    135
결산실적공시예고(안내공시)                                    120
매출액또는손익구조30%(대규모법인은15%)이상변경(자회사의 주요경영사항)          106
연결재무제표기준영업실적등에대한전망(공정공시)                           49
[기재정정]매출액또는손익구조30%(대규모법인은15%)이상변경                  32
[기재정정]매출액또는손익구조30%(대규모법인은15%)이상변동                  29
결산실적공시예고                                           24
영업실적등에대한전망(공정공시)                                   20
매출액또는손익구조30%(대규모법인은15%)이상변동(자회사의 주요경영사항)           18
매출액또는손익구조30%(대규모법인은15%)미만변동(자율공시)                  17
매출액또는손익구조30%(대규모법인15%)미만변경(자율공시)                   15
[기재정정]연결재무제표기준영업(잠정)실적(공정공시)                       12
연결재무제표기준영업(잠정)실적(공정공시)(자회사의 주요경영사항)                 8
[기재정정]영업(잠정)실적(공정공시)                                7
자본잠식50%이상또는매출액50억원미만사실발생(자회사의 주요경영사항)               6
[기재정정]연결재무제표기준영업실적

In [35]:
# KOSPI = Y, KOSDAQ = K
dart_list = dart_list[dart_list["corp_cls"].isin(["Y", "K"])].copy()

keywords = [
    r"영업\(잠정\)실적",
    r"연결재무제표기준영업",
]

pattern = "|".join(keywords)

earnings_disclosures = dart_list[
    dart_list["report_nm"].str.contains(pattern, na=False, regex=True)
].copy()

earnings_disclosures = earnings_disclosures.drop_duplicates(
    subset=["rcept_no"]
).reset_index(drop=True)

earnings_disclosures["market"] = earnings_disclosures["corp_cls"].map({
    "Y": "KOSPI",
    "K": "KOSDAQ"
})

df = earnings_disclosures[[
    "corp_code",
    "corp_name",
    "stock_code",
    "corp_cls",
    "market",
    "report_nm",
    "rcept_no",
    "flr_nm",
    "rcept_dt",
    "rm"
]].copy()

print(df.shape)
df.head()

(16218, 10)


,corp_code,corp_name,stock_code,corp_cls,market,report_nm,rcept_no,flr_nm,rcept_dt,rm
0,01267170,SK케미칼,285130,Y,KOSPI,연결재무제표기준영업(잠정)실적(공정공시),20190131800987,SK케미칼,20190131,유
1,01267170,SK케미칼,285130,Y,KOSPI,영업(잠정)실적(공정공시),20190131800982,SK케미칼,20190131,유
2,00684714,풍산,103140,Y,KOSPI,연결재무제표기준영업(잠정)실적(공정공시),20190131800937,풍산,20190131,유
3,00684714,풍산,103140,Y,KOSPI,영업(잠정)실적(공정공시),20190131800924,풍산,20190131,유
4,00657002,에이디테크놀로지,200710,K,KOSDAQ,연결재무제표기준영업(잠정)실적(공정공시),20190131900857,에이디테크놀로지,20190131,코


In [36]:
df.to_csv(
    "dart_earnings_disclosure_list_2019_2026.csv",
    index=False,
    encoding="utf-8-sig"
)

print(len(df))
df.head()

16218


,corp_code,corp_name,stock_code,corp_cls,market,report_nm,rcept_no,flr_nm,rcept_dt,rm
0,01267170,SK케미칼,285130,Y,KOSPI,연결재무제표기준영업(잠정)실적(공정공시),20190131800987,SK케미칼,20190131,유
1,01267170,SK케미칼,285130,Y,KOSPI,영업(잠정)실적(공정공시),20190131800982,SK케미칼,20190131,유
2,00684714,풍산,103140,Y,KOSPI,연결재무제표기준영업(잠정)실적(공정공시),20190131800937,풍산,20190131,유
3,00684714,풍산,103140,Y,KOSPI,영업(잠정)실적(공정공시),20190131800924,풍산,20190131,유
4,00657002,에이디테크놀로지,200710,K,KOSDAQ,연결재무제표기준영업(잠정)실적(공정공시),20190131900857,에이디테크놀로지,20190131,코


### 02 Minute-Level Timestamp Crawling

In [ ]:
import re
import time
import math
import requests
import pandas as pd
from bs4 import BeautifulSoup
from tqdm.auto import tqdm
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

INPUT = "dart_earnings_disclosure_list_2019_2026.csv"

df = pd.read_csv(INPUT, dtype=str)
df["rcept_dt"] = df["rcept_dt"].str.replace("-", "", regex=False)

target_dates = sorted(df["rcept_dt"].dropna().unique())

session = requests.Session()

retry = Retry(
    total=3,
    connect=3,
    read=3,
    backoff_factor=1.0,
    status_forcelist=[429, 500, 502, 503, 504],
    allowed_methods=["GET", "POST"],
)

adapter = HTTPAdapter(
    max_retries=retry,
    pool_connections=5,
    pool_maxsize=5
)

session.mount("https://", adapter)
session.mount("http://", adapter)

session.headers.update({
    "User-Agent": "Mozilla/5.0",
    "Referer": "https://dart.fss.or.kr/dsac001/mainAll.do",
    "Cookie": "DSAC001_MAXRESULTS=1000;",
})

def yyyymmdd_to_dot(date_str):
    return f"{date_str[:4]}.{date_str[4:6]}.{date_str[6:8]}"

def parse_time_map(html):
    soup = BeautifulSoup(html, "lxml")
    time_map = {}

    rows = soup.select("table tbody tr")

    for tr in rows:
        tds = tr.find_all("td")

        if len(tds) < 3:
            continue

        time_str = tds[0].get_text(strip=True)
        rcept_no = None

        for a in tr.find_all("a"):
            text = " ".join([
                a.get("href", ""),
                a.get("onclick", ""),
                str(a),
            ])

            m = re.search(r"rcpNo[=']+(\d{14})", text)

            if not m:
                m = re.search(r"openReportViewer\('(\d{14})'", text)

            if m:
                rcept_no = m.group(1)
                break

        if rcept_no:
            time_map[rcept_no] = time_str

    return time_map

def get_total_count(html):
    soup = BeautifulSoup(html, "lxml")
    text = soup.get_text(" ", strip=True)

    m = re.search(r"총\s*([\d,]+)\s*건", text)

    if m:
        return int(m.group(1).replace(",", ""))

    return None

def fetch_dart_times_all_pages(date_str):
    select_date = yyyymmdd_to_dot(date_str)

    all_map = {}

    url_main = "https://dart.fss.or.kr/dsac001/mainAll.do"

    res = session.get(
        url_main,
        params={
            "selectDate": select_date,
            "mdayCnt": 0,
        },
        timeout=30,
    )

    res.raise_for_status()

    all_map.update(parse_time_map(res.text))

    total_count = get_total_count(res.text)

    if total_count:
        total_pages = math.ceil(total_count / 100)
    else:
        total_pages = 1

    url_search = "https://dart.fss.or.kr/dsac001/search.ax"

    for page_no in range(2, total_pages + 1):
        res = session.post(
            url_search,
            data={
                "currentPage": page_no,
                "selectDate": select_date,
                "mdayCnt": 0,
            },
            timeout=30,
        )

        res.raise_for_status()

        all_map.update(parse_time_map(res.text))

        time.sleep(0.3)

    return all_map

# Test on a single date before running the full process
test = fetch_dart_times_all_pages("20190131")

print("테스트 수집 건수:", len(test))

if len(test) < 500:
    raise RuntimeError("테스트 수집 실패. 10~30분 후 다시 실행하세요.")

all_time_map = {}
failed_dates = []

for date_str in tqdm(target_dates):
    try:
        time_map = fetch_dart_times_all_pages(date_str)
        all_time_map.update(time_map)

    except Exception as e:
        print(f"[실패] {date_str}: {e}")
        failed_dates.append(date_str)

    time.sleep(0.7)

print("1차 실패 날짜 수:", len(failed_dates))

retry_failed_dates = []

for date_str in tqdm(failed_dates):
    try:
        time_map = fetch_dart_times_all_pages(date_str)
        all_time_map.update(time_map)

    except Exception as e:
        print(f"[재시도 실패] {date_str}: {e}")
        retry_failed_dates.append(date_str)

    time.sleep(1.5)

df["rcept_time"] = df["rcept_no"].map(all_time_map)

success_count = df["rcept_time"].notna().sum()
fail_count = df["rcept_time"].isna().sum()

success_rate = success_count / len(df) * 100
fail_rate = fail_count / len(df) * 100

print("\n===== 최종 결과 =====")
print(f"전체 건수: {len(df):,}")
print(f"시각 수집 성공: {success_count:,} ({success_rate:.2f}%)")
print(f"시각 수집 실패: {fail_count:,} ({fail_rate:.2f}%)")
print(f"최종 실패 날짜 수: {len(retry_failed_dates):,}")

테스트 수집 건수: 630


  0%|          | 0/1286 [00:00<?, ?it/s]

1차 실패 날짜 수: 0


0it [00:00, ?it/s]


===== 최종 결과 =====
전체 건수: 16,218
시각 수집 성공: 16,196 (99.86%)
시각 수집 실패: 22 (0.14%)
최종 실패 날짜 수: 0


In [2]:
df.to_csv(
    "dart_earnings_time_2019_2026.csv",
    index=False,
    encoding="utf-8-sig"
)

### 03 Operating Profit Data Collection

In [ ]:
import requests
import pandas as pd
from bs4 import BeautifulSoup
from tqdm.auto import tqdm
import time
import re
import random
import zipfile
import io
import os

from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry


DART_API_KEY = os.getenv("DART_API_KEY")

INPUT_FILE = "dart_earnings_time_2019_2026.csv"

# Run the full process
SAMPLE_SIZE = None

# Intermediate save filename
CHECKPOINT_FILE = "dart_earnings_financials_2019_2026_checkpoint.csv"
SAVE_EVERY = 100


df = pd.read_csv(INPUT_FILE, dtype=str)

if SAMPLE_SIZE is not None:
    df_run = df.head(SAMPLE_SIZE).copy()
else:
    df_run = df.copy()


# ── Resume from checkpoint ──────────────────────────────────────────
RESULT_COLS = [
    "rcept_no", "sales", "operating_income", "net_income",
    "amount_unit", "sales_won", "operating_income_won",
    "net_income_won", "status", "reason",
]

if os.path.exists(CHECKPOINT_FILE):
    df_checkpoint = pd.read_csv(CHECKPOINT_FILE, dtype=str)

    # Mark only rows with an actual status value (successfully collected) as done
    df_done = df_checkpoint[df_checkpoint["status"].notna() & (df_checkpoint["status"] != "")]
    done_rcept_nos = set(df_done["rcept_no"].dropna().tolist())
    all_results = df_done[RESULT_COLS].to_dict("records")

    df_run = df_run[~df_run["rcept_no"].isin(done_rcept_nos)].copy()
    print(f"[체크포인트 발견] 이미 처리된 건수: {len(done_rcept_nos):,}건 → 이어서 수집합니다.")
    print(f"[남은 수집 대상] {len(df_run):,}건")
else:
    all_results = []
    print("[체크포인트 없음] 처음부터 수집합니다.")
# ────────────────────────────────────────────────────────────────


session = requests.Session()

retry = Retry(
    total=5,
    connect=5,
    read=5,
    backoff_factor=1.5,
    status_forcelist=[429, 500, 502, 503, 504],
    allowed_methods=["GET"],
)

adapter = HTTPAdapter(
    max_retries=retry,
    pool_connections=10,
    pool_maxsize=10,
)

session.mount("https://", adapter)
session.mount("http://", adapter)

session.headers.update({
    "User-Agent": "Mozilla/5.0",
    "Referer": "https://opendart.fss.or.kr/",
})


def parse_number(text):
    if text is None:
        return None

    text = str(text).strip()

    if text in ["", "-", "－", "–", "—"]:
        return None

    negative = False

    if text.startswith("(") and text.endswith(")"):
        negative = True

    if "△" in text or "▲" in text:
        negative = True

    text = (
        text.replace(",", "")
            .replace("원", "")
            .replace("천", "")
            .replace("백만", "")
            .replace("억", "")
            .replace(" ", "")
            .replace("△", "")
            .replace("▲", "")
            .replace("(", "")
            .replace(")", "")
    )

    m = re.search(r"-?\d+(\.\d+)?", text)

    if not m:
        return None

    try:
        value = float(m.group())

        if negative and value > 0:
            value = -value

        if value.is_integer():
            return int(value)

        return value

    except Exception:
        return None


def amount_to_won(value, unit):
    if pd.isna(value):
        return None

    try:
        value = float(value)
    except Exception:
        return None

    unit = str(unit).replace(" ", "") if unit is not None else ""

    if unit == "억원":
        return int(value * 100_000_000)

    if unit == "백만원":
        return int(value * 1_000_000)

    if unit == "천원":
        return int(value * 1_000)

    if unit == "원":
        return int(value)

    return None


def safe_get(url, params=None, timeout=30, max_try=5):
    for i in range(max_try):
        try:
            res = session.get(url, params=params, timeout=timeout)
            res.raise_for_status()
            return res

        except requests.exceptions.RequestException as e:
            print(f"[요청 실패] {i + 1}/{max_try} | {url} | {e}")
            time.sleep(2 + random.random() * 3)

    return None


def get_document_html_from_opendart(rcept_no):
    url = "https://opendart.fss.or.kr/api/document.xml"

    params = {
        "crtfc_key": API_KEY,
        "rcept_no": rcept_no,
    }

    res = safe_get(url, params=params)

    if res is None:
        return None, "document request failed"

    content = res.content

    if not content.startswith(b"PK"):
        try:
            res.encoding = "utf-8"
            soup = BeautifulSoup(res.text, "xml")

            status = soup.find("status")
            message = soup.find("message")

            status_text = status.get_text(strip=True) if status else None
            message_text = message.get_text(strip=True) if message else None

            return None, f"OpenDART error: status={status_text}, message={message_text}"

        except Exception:
            return None, f"not zip response: {res.text[:500]}"

    try:
        zf = zipfile.ZipFile(io.BytesIO(content))
        texts = []

        for name in zf.namelist():
            raw = zf.read(name)

            decoded = None

            for enc in ["utf-8", "cp949", "euc-kr"]:
                try:
                    decoded = raw.decode(enc)
                    break
                except UnicodeDecodeError:
                    continue

            if decoded is None:
                decoded = raw.decode("utf-8", errors="ignore")

            texts.append(decoded)

        return "\n".join(texts), None

    except Exception as e:
        return None, f"zip parse failed: {e}"


def extract_unit(text):
    compact = re.sub(r"\s+", "", text)

    if "단위:억원" in compact:
        return "억원"

    if "단위:백만원" in compact:
        return "백만원"

    if "단위:천원" in compact:
        return "천원"

    if "단위:원" in compact:
        return "원"

    return None


def is_bad_value_cell(text):
    text = str(text)

    bad_keywords = [
        "%",
        "증감율",
        "증감률",
        "비율",
        "율",
        "전년",
        "전기대비",
    ]

    return any(k in text for k in bad_keywords)


def find_metric_value_from_tables(html, metric_keywords):
    soup = BeautifulSoup(html, "lxml")

    for tr in soup.find_all("tr"):
        cells = tr.find_all(["td", "th"])

        if len(cells) < 2:
            continue

        cell_texts = [c.get_text(" ", strip=True) for c in cells]
        row_text = " ".join(cell_texts)

        if not any(k in row_text for k in metric_keywords):
            continue

        metric_idx = None

        for i, txt in enumerate(cell_texts):
            if any(k in txt for k in metric_keywords):
                metric_idx = i
                break

        if metric_idx is None:
            continue

        for txt in cell_texts[metric_idx + 1:]:
            if is_bad_value_cell(txt):
                continue

            value = parse_number(txt)

            if value is not None:
                return value

    return None


def find_metric_value_from_text(html, metric_keywords):
    soup = BeautifulSoup(html, "lxml")
    text = soup.get_text(" ", strip=True)

    for keyword in metric_keywords:
        pattern = keyword + r".{0,80}?([△▲(]?\s*[\d,]+(?:\.\d+)?\s*[)]?)"
        m = re.search(pattern, text)

        if m:
            return parse_number(m.group(1))

    return None


def fetch_financials(rcept_no):
    result = {
        "rcept_no": rcept_no,
        "sales": None,
        "operating_income": None,
        "net_income": None,
        "amount_unit": None,
        "sales_won": None,
        "operating_income_won": None,
        "net_income_won": None,
        "status": "fail",
        "reason": None,
    }

    html, err = get_document_html_from_opendart(rcept_no)

    if html is None:
        result["reason"] = err
        return result

    result["amount_unit"] = extract_unit(html)

    result["sales"] = find_metric_value_from_tables(html, ["매출액"])

    result["operating_income"] = find_metric_value_from_tables(html, ["영업이익", "영업손실"])

    result["net_income"] = find_metric_value_from_tables(
        html,
        ["당기순이익", "당기순손실", "분기순이익", "분기순손실", "반기순이익", "반기순손실"]
    )

    if result["sales"] is None:
        result["sales"] = find_metric_value_from_text(html, ["매출액"])

    if result["operating_income"] is None:
        result["operating_income"] = find_metric_value_from_text(html, ["영업이익", "영업손실"])

    if result["net_income"] is None:
        result["net_income"] = find_metric_value_from_text(
            html,
            ["당기순이익", "당기순손실", "분기순이익", "분기순손실", "반기순이익", "반기순손실"]
        )

    result["sales_won"] = amount_to_won(result["sales"], result["amount_unit"])
    result["operating_income_won"] = amount_to_won(result["operating_income"], result["amount_unit"])
    result["net_income_won"] = amount_to_won(result["net_income"], result["amount_unit"])

    if any([
        result["sales"] is not None,
        result["operating_income"] is not None,
        result["net_income"] is not None,
    ]):
        result["status"] = "success"
    else:
        result["reason"] = "target values not found"

    return result


# ── Data collection loop ────────────────────────────────────────────
for i, rcept_no in enumerate(tqdm(df_run["rcept_no"], total=len(df_run)), start=1):
    result = fetch_financials(rcept_no)
    all_results.append(result)

    if i % SAVE_EVERY == 0:
        temp_financials = pd.DataFrame(all_results)
        temp_df = df.merge(temp_financials, on="rcept_no", how="left")

        temp_df.to_csv(CHECKPOINT_FILE, index=False, encoding="utf-8-sig")
        print(f"[중간 저장] 누적 {len(all_results):,}건 완료 → {CHECKPOINT_FILE}")

    time.sleep(random.uniform(0.3, 0.8))
# ────────────────────────────────────────────────────────────────


# ── Merge final results ─────────────────────────────────────────────
financials_df = pd.DataFrame(all_results)

df_final = df.merge(financials_df, on="rcept_no", how="left")
# ────────────────────────────────────────────────────────────────

print("===== 03 영업이익 수치 전체 수집 결과 =====")
print(f"총 대상 건수: {len(df_final):,}")
print(f"성공 건수: {(df_final['status'] == 'success').sum():,}")
print(f"매출액 수집 성공: {df_final['sales'].notna().sum():,}")
print(f"영업이익 수집 성공: {df_final['operating_income'].notna().sum():,}")
print(f"순이익 수집 성공: {df_final['net_income'].notna().sum():,}")
print(f"원 단위 영업이익 생성 성공: {df_final['operating_income_won'].notna().sum():,}")

print(
    df_final[
        [
            "corp_name",
            "rcept_no",
            "report_nm",
            "rcept_dt",
            "rcept_time",
            "sales",
            "operating_income",
            "net_income",
            "amount_unit",
            "sales_won",
            "operating_income_won",
            "net_income_won",
            "status",
            "reason",
        ]
    ].head(20)
)

[체크포인트 발견] 이미 처리된 건수: 500건 → 이어서 수집합니다.
[남은 수집 대상] 15,718건


  0%|          | 0/15718 [00:00<?, ?it/s]

[중간 저장] 누적 600건 완료 → dart_earnings_financials_2019_2026_checkpoint.csv
[중간 저장] 누적 700건 완료 → dart_earnings_financials_2019_2026_checkpoint.csv
[중간 저장] 누적 800건 완료 → dart_earnings_financials_2019_2026_checkpoint.csv
[중간 저장] 누적 900건 완료 → dart_earnings_financials_2019_2026_checkpoint.csv
[중간 저장] 누적 1,000건 완료 → dart_earnings_financials_2019_2026_checkpoint.csv
[중간 저장] 누적 1,100건 완료 → dart_earnings_financials_2019_2026_checkpoint.csv
[중간 저장] 누적 1,200건 완료 → dart_earnings_financials_2019_2026_checkpoint.csv
[중간 저장] 누적 1,300건 완료 → dart_earnings_financials_2019_2026_checkpoint.csv
[중간 저장] 누적 1,400건 완료 → dart_earnings_financials_2019_2026_checkpoint.csv
[중간 저장] 누적 1,500건 완료 → dart_earnings_financials_2019_2026_checkpoint.csv
[중간 저장] 누적 1,600건 완료 → dart_earnings_financials_2019_2026_checkpoint.csv
[중간 저장] 누적 1,700건 완료 → dart_earnings_financials_2019_2026_checkpoint.csv
[중간 저장] 누적 1,800건 완료 → dart_earnings_financials_2019_2026_checkpoint.csv
[중간 저장] 누적 1,900건 완료 → dart_earnings_financials_2019_2026_c

In [4]:
df_final.to_csv(
    "dart_earnings_financials_2019_2026.csv",
    index=False,
    encoding="utf-8-sig"
)

## 1. Disclosure Data and Registration Timestamp Collection

### 1-1. DART Disclosure List Collection

Disclosure data and registration timestamps were collected using the Financial Supervisory Service's DART OpenAPI. Python's `requests` and `OpenDartReader` were used, and the analysis period was set from January 2019 to April 2026.

First, monthly disclosure lists were collected using DART's `list.json` API. The disclosure type was set to `pblntf_ty="I"`, which includes major event reports and earnings-related disclosures. All pages for each month were retrieved iteratively to ensure complete coverage of the disclosure list.

Among the collected disclosures, the analysis was restricted to firms listed on KOSPI and KOSDAQ. Based on DART's `corp_cls` classification, KOSPI-listed firms were identified as `Y` and KOSDAQ-listed firms as `K`.

Next, only disclosures whose report names (`report_nm`) contained the following keywords were extracted:

```text
영업(잠정)실적
연결재무제표기준영업
```

This filtering process was used to identify disclosures related to quarterly or periodic operating results. When duplicate records shared the same receipt number (`rcept_no`), only one disclosure was retained.

The final disclosure list dataset contained the following variables:

```text
corp_code
corp_name
stock_code
corp_cls
market
report_nm
rcept_no
flr_nm
rcept_dt
rm
```

The resulting dataset was saved as:

```text
dart_earnings_disclosure_list_2019_2026.csv
```

---

### 1-2. Minute-Level Disclosure Registration Timestamp Collection

Although the DART OpenAPI disclosure list provides the receipt date (`rcept_dt`), it does not directly provide the exact registration time at the minute level. Therefore, the DART website was additionally crawled to collect the actual registration time (`rcept_time`) for each disclosure.

First, the relevant receipt dates were extracted from the disclosure list dataset. The DART full disclosure page was then queried for each date. Receipt numbers (`rcept_no`) and registration times were parsed from the HTML tables and matched with the disclosure list dataset using the receipt number as the key.

To improve the stability and reliability of the crawling process, the following procedures were implemented:

```text
1. Use of a requests Session
2. HTTP retry configuration
3. User-Agent and Referer configuration
4. Iterative collection of all pages for each date
5. Retry of failed dates
6. Verification of collection success rates and failed records
```

Because daily disclosures may span multiple pages, the total number of disclosures was first identified from the first page, after which all corresponding pages were retrieved iteratively.

The minute-level registration time was then merged with the disclosure list using `rcept_no`, creating the `rcept_time` variable.

The resulting dataset was saved as:

```text
dart_earnings_time_2019_2026.csv
```

---

### 1-3. Operating Performance Data Collection

In addition to disclosure timing information, key financial figures from each earnings disclosure were collected to classify disclosures as good news or bad news.

The collected variables related to operating performance included:

```text
sales
operating_income
net_income
sales_won
operating_income_won
net_income_won
amount_unit
```

In particular, because the classification of good and bad news in this study was based on the quarter-over-quarter change in operating income, `operating_income_won` was used as the primary variable for this classification.

The disclosure information, minute-level registration timestamps, and operating performance figures were then combined to construct the raw dataset for subsequent analysis.

The resulting dataset was saved as:

```text
dart_earnings_financials_2019_2026.csv
```

---

### 1-4. Use of the Collected Data

The dataset constructed through this process was used in subsequent analyses as follows:

| Data | Purpose |
|---|---|
| Report name and receipt number | Identification of earnings disclosures |
| Receipt date | Construction of year, day-of-week, and event-date variables |
| Minute-level registration time | Classification of pre-market, intraday, and after-hours disclosures |
| Market classification | Comparison between KOSPI and KOSDAQ |
| Operating income | Classification of good news and bad news |
| Stock code | Merging with stock price, financial, and firm-characteristic data |

Overall, this stage constructed the core dataset underlying the subsequent disclosure timing pattern analysis, Event Study, and regression analysis.